# 🚀 AIE4ML Tutorial 2: Branching + Transpose + Multi-output

This short tutorial focuses on one model that stresses key architectural features:
- internal fanout / broadcasting after a shared dense layer
- a `Transpose` in one branch
- two output tensors
- ND dense path (Dense on rank-3 tensors)

## ⚙️ Setup & Imports

```bash
conda create -n aie4ml python=3.11 -y && conda activate aie4ml
pip install "aie4ml[onnx]" onnxruntime ipykernel
```

In [ ]:
import numpy as np
import onnx
import onnxruntime as ort
from onnx import TensorProto, helper, numpy_helper

from aie4ml.frontends.onnx import from_onnx

## 1️⃣ The model

WIth this model we exercise more features of the aie4ml compiler. A shared trunk fans out to two branches. **Branch 1** applies a `Transpose`; **Branch 2** is a
straight dense chain. Each branch has its own output. The input is quantized to **16 bits** while weights and activations are int8.

The model is a quantized (QDQ) ONNX graph: each `QuantizeLinear`/`DequantizeLinear` pair fixes a tensor's
fixed-point type, which aie4ml implements exactly.

In [ ]:
TOKENS, FEATURES = 16, 64
rng = np.random.default_rng(0)
nodes, inits = [], []

def qparams(name, frac, dtype):
    inits.extend([numpy_helper.from_array(np.float32(2.0**-frac), f"{name}_scale"),
                  helper.make_tensor(f"{name}_zp", dtype, [], [0])])
    return [f"{name}_scale", f"{name}_zp"]

def quantize(x, name, frac, dtype=TensorProto.INT8):
    """Quantize -> dequantize: pins the tensor to a fixed-point type with `frac` fractional bits."""
    params = qparams(name, frac, dtype)
    nodes.append(helper.make_node("QuantizeLinear", [x, *params], [f"{name}_q"]))
    nodes.append(helper.make_node("DequantizeLinear", [f"{name}_q", *params], [name]))
    return name

def dense(x, name, n_in, n_out, relu=True):
    """int8 weights (frac 7); the output is int8, or uint8 after a ReLU."""
    inits.append(numpy_helper.from_array(rng.integers(-64, 64, (n_in, n_out), dtype=np.int8), f"{name}_wq"))
    nodes.append(helper.make_node("DequantizeLinear", [f"{name}_wq", *qparams(f"{name}_w", 7, TensorProto.INT8)], [f"{name}_w"]))
    nodes.append(helper.make_node("MatMul", [x, f"{name}_w"], [f"{name}_mm"], name=name))
    if not relu:
        return quantize(f"{name}_mm", f"{name}_out", 2)
    nodes.append(helper.make_node("Relu", [f"{name}_mm"], [f"{name}_relu"]))
    return quantize(f"{name}_relu", f"{name}_out", 5, TensorProto.UINT8)

nodes.append(helper.make_node("DequantizeLinear", ["x", *qparams("x", 9, TensorProto.INT16)], ["x_dq"]))
trunk = dense("x_dq", "trunk_fc", FEATURES, FEATURES)

b1 = dense(trunk, "b1_fc1", FEATURES, FEATURES)
nodes.append(helper.make_node("Transpose", [b1], ["b1_t"], perm=[0, 2, 1]))       # (1, FEATURES, TOKENS)
b1 = dense("b1_t", "b1_fc2", TOKENS, FEATURES, relu=False)

b2 = dense(trunk, "b2_fc1", FEATURES, FEATURES)
b2 = dense(b2, "b2_fc2", FEATURES, FEATURES)
b2 = dense(b2, "b2_fc3", FEATURES, FEATURES, relu=False)

graph = helper.make_graph(
    nodes, "branching_model",
    [helper.make_tensor_value_info("x", TensorProto.INT16, [1, TOKENS, FEATURES])],
    [helper.make_tensor_value_info(b1, TensorProto.FLOAT, [1, FEATURES, FEATURES]),
     helper.make_tensor_value_info(b2, TensorProto.FLOAT, [1, TOKENS, FEATURES])],
    initializer=inits,
)
model = helper.make_model(graph, opset_imports=[helper.make_operatorsetid("", 21)])
onnx.checker.check_model(model)

## 2️⃣ Tuning Directives

Directives are attached **per layer** in the input configuration. The same configuration is also supported in hls4ml frontend through the input config.

### 🔀 Parallelism

`cas_num` specifies the number of **CAS chains operating in parallel**, with each chain computing a portion of the output:

- `contract: 'inner'` → splits the **output features**
- `contract: 'outer'` → splits the **rows**

Each chain is `cas_length` tiles deep, thereby splitting the reduction across the chains.

> **Directives are constraints, not complete specifications.**  
> The compiler (`optimize`, as introduced in Tutorial 1) preserves what a directive explicitly fixes and chooses the remaining parameters automatically. Therefore, a directive may specify only `cas_num`, for example.

### 🚚 Routing

`io_route` determines how a tensor is routed to a layer:

- `'direct'` → tile-to-tile, or PLIO-to-tile routing
- `'memtile'` → routes the tensor through a memory tile (on **AIE-ML** and **AIE-MLv2**)

The compiler uses **direct routing whenever possible** and inserts a memory tile only when the data needs to be reordered, zero-padded or we need more DMA buffer descriptors than a tile has.

`io_route` overrides the choice: `'direct'` guarantees no memory tile (the configuration is refused otherwise), `'memtile'` forces one. The routing is keyed by the layer's **input tensor name**; if an invalid name is provided, the configuration is rejected and the valid input names are listed.

### 📍 Placement

`{'row': r, 'col': c}` pins a kernel (or the southwest tile of a graph) to a specific AIE tile as its starting point. In general, the compiler's **automatic placer** can usually determine a suitable placement for us to minimize hops between graphs.

### 🎯 Optimization mode (covered in Tutorial 1)

Layers without directives are split by the compiler. `"Optimize": "resource"` (default) uses the fewest tiles;
`"Optimize": "performance"` adds parallelism, up to `"MaxTiles"`, to reduce the work on the busiest tile. Below,
the trunk and branch 2 are fixed by directives, branch 1 is left to the compiler.

In [ ]:
directives = {}
for name in ("trunk_fc", "b2_fc1", "b2_fc2", "b2_fc3"):
    directives[name] = {"parallelism": {"cas_num": 2, "cas_length": 1, "contract": "outer"}}

for column, name in zip((7, 8, 9), ("b2_fc1", "b2_fc2", "b2_fc3")):
    directives[name]["placement"] = {"row": 1, "col": column}

# Route branch 1's input, the trunk's ReLU output, through a memory tile.
directives["b1_fc1"] = {"io_route": {"inputs": {"trunk_fc_relu": "memtile"}}}

## 3️⃣ Convert and compile

Convert with the ONNX frontend, then compile for the x86 simulator.

In [ ]:
aie_model = from_onnx(
    model,
    {
        "Part": "xilinx_vek280_base",
        "AIEConfig": {"Iterations": 10, "Optimize": "performance", "MaxTiles": 10},
        "LayerDirectives": directives,
    },
    output_dir="proj_aie_t2",
    project_name="proj_aie_t2",
)
aie_model.write()


The array layout shows each kernel graph on its tiles, even before compilation

In [ ]:
from aie4ml.report import layout
layout(aie_model)

## 4️⃣ Bit-exact comparison with onnxruntime


In [ ]:
#compile model in x86 compilation
aie_model.compile()

x = rng.integers(-512, 512, (1, TOKENS, FEATURES), dtype=np.int16)

options = ort.SessionOptions()
options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_BASIC  # the graph's own QDQ arithmetic
session = ort.InferenceSession(model.SerializeToString(), options, providers=["CPUExecutionProvider"])
y_ref = dict(zip([o.name for o in session.get_outputs()], session.run(None, {"x": x})))

y_aie = aie_model.predict({"x": x}, simulator="x86")
for name, ref in y_ref.items():
    print(f"{name}: max abs diff = {float(np.max(np.abs(np.asarray(y_aie[name]) - ref)))}")

## 5️⃣ Build for the AI Engine and profile

The x86 simulator checks the output numbers but not the timing. To get **cycles and latency**, we
compile the graph for the AI Engine and run the cycle-accurate simulator with profiling.

In [ ]:
aie_model.build()
y_aie_sim = aie_model.predict({"x": x}, simulator="aie")

for name, ref in y_ref.items():
    print(f"{name}: max abs diff = {float(np.max(np.abs(np.asarray(y_aie_sim[name]) - ref)))}")

## 6️⃣ The project report

`report` summarises the compiled project — how many kernels, how many AIE tiles and memory-tile
buffers, and (after running inference in '`aie`' mode) cycles and latency. 

In [ ]:
from aie4ml.report import report

# Takes a model or a project directory -- report('proj_aie_t2') works the same.
report(aie_model)

## 💡 Takeaways

- One model exercised **fanout, a transpose, two outputs, ND dense, and a 16-bit input** 
- **Parallelism** (`cas_num` / `cas_length` / `contract`) trades AIE tiles for throughput.
- A **transpose** is realised through a memory tile automatically.
- Tuning doesn't change the numerics — the result stays **bit-exact** with onnxruntime.

The same directives apply through the hls4ml frontend.